# Metrics

In [ ]:
from sklearn.metrics import confusion_matrix
y_true = [1,0,0,1,1,0,1,1]
y_pred = [1,0,0,1,0,1,1,0]
cm = confusion_matrix(y_true, y_pred)

TN, FP, FN, TP = cm.ravel()

print(f"""
     P   N
     O   E
     S   G
POS  {TP}   {FN}
NEG  {FP}   {TN}
""")
import sklearn.metrics
acc = sklearn.metrics.accuracy_score(y_true, y_pred)
accuracy = (TP + TN) / cm.sum()

acc, accuracy

In [ ]:
import numpy as np
y_true = np.random.choice(3, 100, replace=True, p=[0.5, 0.3, 0.2])
y_pred = np.random.choice(3, 100, replace=True, p=[0.5, 0.3, 0.2])
confusion_matrix(y_true, y_pred)

In [ ]:
y_true = np.random.choice(2, 100, replace=True, p=[0.7, 0.3])
y_pred = np.ones(100)
cm = confusion_matrix(y_true, y_pred)
TN, FP, FN, TP = cm.ravel()
accuracy = (TP + TN) / cm.sum()
accuracy

In [ ]:
y_true = np.random.choice(2, 100, replace=True, p=[0.1, 0.9])
y_pred = np.ones(100)
cm = confusion_matrix(y_true, y_pred)
TN, FP, FN, TP = cm.ravel()
accuracy = (TP + TN) / cm.sum()
TPR = TP / (TP + FN)
TNR = TN / (TN + FP)

BA = (TPR + TNR) / 2
BA = sklearn.metrics.balanced_accuracy_score(y_true, y_pred)

F1 = 2 * TP / (2 * TP + FP + FN)
print(f"accuracy: {accuracy}\nTPR: {TPR}\nTNR : {TNR}\nBalanced accuracy: {BA}\nF1: {F1}")

precision = TP / (TP + FP)
recall = TP / (TP + FN)
print(f"precision = {precision}\nrecall={recall}\nF1={2 * precision * recall / (precision + recall)}")

F1 = sklearn.metrics.f1_score(y_true, y_pred)

Here are some measures for evaluating multiclass classifiers:
- **f1_macro** Calculate mean of metric scores for each class, weighting each class equally
- **f1_micro** Calculate mean of metric scores for each observation-class combination
- **f1_weighted** Calculate mean of metric scores for each class, weighting each class proportional to its size in the data

In [ ]:
from sklearn import metrics
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
import pandas as pd
url="https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
wine = pd.read_csv(url, sep = ";")
features = wine.drop("quality", axis = 1)
target = wine["quality"]
target.value_counts()

# Cross validation

In [ ]:
from sklearn.ensemble import RandomForestClassifier
standardizer = StandardScaler()
classifier = KNeighborsClassifier()
#classifier = RandomForestClassifier()
pipeline = make_pipeline(standardizer, classifier)
kf = KFold(n_splits=10, shuffle=True, random_state=1)
cv_results = cross_val_score(pipeline, # Pipeline
  features, # Feature matrix
  target, # Target vector
  cv=kf, # Cross-validation technique
  scoring="f1_weighted", # Loss function
  n_jobs=-1) # Use all CPU scores
# Calculate mean
cv_results.mean()

# All possible metrics
# sorted(sklearn.metrics.SCORERS.keys())
#

# Classification report

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
features_train, features_test, target_train, target_test = train_test_split(features, target)
model = classifier.fit(features_train, target_train)
target_predicted = model.predict(features_test)
# Create a classification report
print(classification_report(target_test, target_predicted, zero_division = 0))

In [ ]:
scaler = StandardScaler()
scaler.fit(features_train)
other_model = classifier.fit(scaler.transform(features_train), target_train)
target_predicted = model.predict(scaler.transform(features_test))
print(classification_report(target_test, target_predicted, zero_division = 0))

# Visualizing effect of hyperparameters

In [ ]:
from sklearn.model_selection import validation_curve
import matplotlib.pyplot as plt
scaler = StandardScaler()
param_range = np.arange(1, 50, 1)
train_scores, test_scores = validation_curve(
  # Classifier
  RandomForestClassifier(),
  # Feature matrix
  features,
  # Target vector
  target,
  # Hyperparameter to examine
  param_name="n_estimators",
  # Range of hyperparameter's values
  param_range = param_range,
  # Number of folds
  cv=3,
  # Performance metric
  scoring="accuracy",
  # Use all computer cores
  n_jobs=-1)
# Calculate mean and standard deviation for training set scores
train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
# Calculate mean and standard deviation for test set scores
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)
# Plot mean accuracy scores for training and test sets
plt.plot(param_range, train_mean, label="Training score", color="black")
plt.plot(param_range, test_mean, label="Cross-validation score", color="dimgrey")
# Plot accurancy bands for training and test sets
plt.fill_between(param_range, train_mean - train_std,
  train_mean + train_std, color="gray")
plt.fill_between(param_range, test_mean - test_std,
  test_mean + test_std, color="gainsboro")
# Create plot
plt.title("Validation Curve With random forest")
plt.xlabel("Number Of trees")
plt.ylabel("Accuracy Score")
plt.tight_layout()
plt.legend(loc="best")
plt.show()

# Grid search for hyperparameters

In [ ]:
import numpy as np
from sklearn import linear_model, datasets
from sklearn.model_selection import GridSearchCV
# Load data
iris = datasets.load_iris()
features = iris.data
target = iris.target
# Create logistic regression
logistic = linear_model.LogisticRegression(solver = "liblinear", max_iter=500)
# Create range of candidate penalty hyperparameter values
penalty = ['l1', 'l2']
# Create range of candidate regularization hyperparameter values
C = np.logspace(0, 4, 10)
# Create dictionary hyperparameter candidates
hyperparameters = dict(C=C, penalty=penalty)
# Create grid search
gridsearch = GridSearchCV(logistic, hyperparameters, cv=5, verbose=0)
# Fit grid search
best_model = gridsearch.fit(features, target)
print('Best Penalty:', best_model.best_estimator_.get_params()['penalty'])
print('Best C:', best_model.best_estimator_.get_params()['C'])

# Careful! Bad science!
target_predicted = best_model.predict(features)
print(classification_report(target, target_predicted, zero_division = 0))

# Randomized search

In [ ]:
from scipy.stats import uniform
from sklearn import linear_model, datasets
from sklearn.model_selection import RandomizedSearchCV
# Load data
iris = datasets.load_iris()
features = iris.data
target = iris.target
# Create logistic regression
logistic = linear_model.LogisticRegression(solver = "liblinear", max_iter=500)
# Create range of candidate regularization penalty hyperparameter values
penalty = ['l1', 'l2']
# Create distribution of candidate regularization hyperparameter values
C = uniform(loc=0, scale=4)
# Create hyperparameter options
hyperparameters = dict(C=C, penalty=penalty)
# Create randomized search
randomizedsearch = RandomizedSearchCV(
  logistic, hyperparameters, random_state=1, n_iter=100, cv=5, verbose=0,
  n_jobs=-1)
# Fit randomized search
best_model = randomizedsearch.fit(features, target)
print('Best Penalty:', best_model.best_estimator_.get_params()['penalty'])
print('Best C:', best_model.best_estimator_.get_params()['C'])

# Careful! Bad science!
target_predicted = best_model.predict(features)
print(classification_report(target, target_predicted, zero_division = 0))

# Selecting Best Models from Multiple Learning Algorithms

In [ ]:
import numpy as np
from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
# Set random seed
np.random.seed(0)
# Load data
iris = datasets.load_iris()
features = iris.data
target = iris.target
# Create a pipeline
pipe = Pipeline([("classifier", RandomForestClassifier())])
# Create dictionary with candidate learning algorithms and their hyperparameters
search_space = [{"classifier": [LogisticRegression(solver = "liblinear", max_iter=500)],
  "classifier__penalty": ['l1', 'l2'],
  "classifier__C": np.logspace(0, 4, 10)},
  {"classifier": [RandomForestClassifier()],
  "classifier__n_estimators": [10, 100, 1000],
  "classifier__max_features": [1, 2, 3]}]
# Create grid search
gridsearch = GridSearchCV(pipe, search_space, cv=5, verbose=0)
# Fit grid search
best_model = gridsearch.fit(features, target)
best_model.best_estimator_.get_params()["classifier"]

# Selecting Best Models When Preprocessing

In [ ]:
import numpy as np
from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
# Set random seed
np.random.seed(0)
# Load data
iris = datasets.load_iris()
features = iris.data
target = iris.target
# Create a preprocessing object that includes StandardScaler features and PCA
preprocess = FeatureUnion([("std", StandardScaler()), ("pca", PCA())])
# Create a pipeline
pipe = Pipeline([("preprocess", preprocess),
("classifier", LogisticRegression(solver = "liblinear", max_iter=500))])
# Create space of candidate values
search_space = [{"preprocess__pca__n_components": [1, 2, 3],
  "classifier__penalty": ["l1", "l2"],
  "classifier__C": np.logspace(0, 4, 10)}]
# Create grid search
clf = GridSearchCV(pipe, search_space, cv=5, verbose=0, n_jobs=-1)
# Fit grid search
best_model = clf.fit(features, target)
best_model.best_estimator_.get_params()

# Nested Cross validation

In [ ]:
import numpy as np
from sklearn import linear_model, datasets
from sklearn.model_selection import GridSearchCV, cross_val_score
# Load data
iris = datasets.load_iris()
features = iris.data
target = iris.target
# Create logistic regression
logistic = linear_model.LogisticRegression(max_iter=500)
# Create range of 20 candidate values for C
C = np.logspace(0, 4, 20)
# Create hyperparameter options
hyperparameters = dict(C=C)
# Create grid search
gridsearch = GridSearchCV(logistic, hyperparameters, cv=5, n_jobs=-1, verbose=0)
# Conduct nested cross-validation and outut the average score
cross_val_score(gridsearch, features, target).mean()

# k Nearest Neighbor

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import GridSearchCV
# Load data
iris = datasets.load_iris()
features = iris.data
target = iris.target
# Create standardizer
standardizer = StandardScaler()
# Standardize features
features_standardized = standardizer.fit_transform(features)
# Create a KNN classifier
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
# Create a pipeline
pipe = Pipeline([("standardizer", standardizer), ("knn", knn)])
# Create space of candidate values
search_space = [{"knn__n_neighbors": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]}]
# Create grid search
classifier = GridSearchCV(pipe, search_space, cv=5, verbose=0).fit(features_standardized, target)
classifier.best_estimator_.get_params()["knn__n_neighbors"]